# Fine-tuning RoBERTa

**Question (Further Work):** how much better does RoBERTa get when we train *all* of its weights for this task, instead of freezing it and training only a classifier on top?

Every transformer notebook so far has been **frozen**. RoBERTa turned each post into a vector once, and only LR or a small MLP learned from those vectors. The best frozen RoBERTa result is 68.33 (the tuned MLP on `<s>`, mean of 10 seeds).

**Fine-tuning** keeps the same pre-trained model and adds a classification head, but now the loss updates every weight inside RoBERTa too. RoBERTa can then change what it puts into `<s>` to suit *this* task. This is the "pre-train, then fine-tune" recipe from Lecture 11.

Same data and test set as the other notebooks: headers, footers and quotes removed, empty posts dropped.

**Predict before running:** frozen RoBERTa + MLP gets 68.33, and the frozen Qwen3-Embedding-4B + MLP gets 78.97. Where will fine-tuned RoBERTa land?

## What changes from the frozen notebooks

1. **RoBERTa's weights go into the optimizer, and its forward pass isn't wrapped in `torch.no_grad()`.** That's the whole difference in code. Gradients now flow from the loss back through the head *and* all 12 of RoBERTa's layers.
2. **A tiny learning rate (2e-5), with warmup.** RoBERTa's weights are already good, so they only need nudging. Big steps would overwrite what pre-training learned.
3. **The RoBERTa paper's fine-tuning recipe** (Liu et al. 2019, Table 10): up to 10 epochs, keeping the epoch with the best dev accuracy, learning rate from {1e-5, 2e-5, 3e-5}, weight decay 0.1, 6% warmup. An earlier version of this notebook used the BERT paper's shorter recipe (3 epochs). `fine-tuning-results.md` compares the two, and `sweep.py` is the script that tuned this one.
4. **Memory.** Backpropagation needs every layer's activations from the forward pass, for every token in the batch. That limits the batch to 16 posts, and it's why we compute in bfloat16.

## Load dataset

In [1]:
## Python imports: "import X as Y" loads library X under the short name Y (a bit like #include plus a namespace alias)
import numpy as np        ## numpy: n-dimensional arrays with math on whole arrays at once
import torch              ## PyTorch: tensors (numpy-like arrays that can live in GPU memory) + automatic gradients
import torch.nn as nn     ## building blocks for neural networks: layers, losses
from sklearn.datasets import fetch_20newsgroups   ## "from X import Y": take just Y out of library X

## Download (the first time only) and load the posts. remove=(...) strips headers, signatures and quoted replies.
## The result works like a struct: .data is a Python list of strings (one post each), .target the class ids (0-19),
## .target_names the 20 newsgroup names, .filenames each post's file.
newsgroups_train = fetch_20newsgroups(
    subset='train', remove=('headers', 'footers', 'quotes'))

newsgroups_test = fetch_20newsgroups(
    subset='test', remove=('headers', 'footers', 'quotes'))

# Remove empty documents
## zip(a, b) walks two lists side by side: first (newsgroups_train, "train"), then (newsgroups_test, "test")
for s, name in zip([newsgroups_train, newsgroups_test], ["train", "test"]):
  ## {expr for ... if ...} builds a set (like std::unordered_set<int>): the indices i of the empty posts.
  ## enumerate(list) yields (index, element) pairs, like a C++ loop that also keeps a counter.
  empty_indices = {i for i, doc in enumerate(s.data) if len(doc) == 0}
  orig_len = len(s.data)
  for k in ['data', 'filenames', 'target']:
    ## [expr for ... if ...] builds a new list: element i of s[k], for every i that isn't an empty post
    s[k] = [s[k][i] for i in range(orig_len) if i not in empty_indices]
  ## f"...{x}..." is a format string: each {x} is replaced by x's value (like std::format)
  print(f"Removed {len(empty_indices)} empty documents from the {name} set. Before: {orig_len}. After: {len(s.data)}.")

y_train = np.array(newsgroups_train.target)      ## (11096,) int64: the class id of each training post
y_test = np.array(newsgroups_test.target)        ## (7370,)
n_classes = len(newsgroups_train.target_names)   ## 20

Removed 218 empty documents from the train set. Before: 11314. After: 11096.
Removed 162 empty documents from the test set. Before: 7532. After: 7370.


## Dev set: 10% of the *threads*

Fine-tuning needs a dev set to choose which epoch's weights to keep. The frozen notebooks held out a random 10% of the posts. Here we hold out 10% of the **threads** instead:
- Replies share their thread's subject line ("Re: ..."). With a random split, most dev posts have a thread-mate in the training part, so the dev score rewards remembering threads rather than learning topics (`Lessons.md`, "Make the folds as independent as train and test are").
- A fine-tuned RoBERTa can memorize far more than an MLP on frozen vectors, so this matters more here.
- The headers are only read to find each post's thread. The model never sees them.

In [2]:
import re                                         ## regular expressions (Lecture 2)
from sklearn.model_selection import StratifiedGroupKFold

## Read train again WITH headers, only to get each post's subject line (11,314 posts, empty ones included)
with_headers = fetch_20newsgroups(subset='train')

## A dict comprehension {key: value for ...} builds a dict (like std::unordered_map<std::string, std::string>):
## file name -> subject line. re.search finds the line that starts with "Subject:" (re.M lets ^ and $ match at
## every line), and .group(1) is the text captured by the (.*) part.
subject_of = {f: re.search(r"^Subject:(.*)$", doc, re.M).group(1)
              for f, doc in zip(with_headers.filenames, with_headers.data)}

## One thread name per post we kept, in the same order as newsgroups_train.data: the subject without any leading
## "Re:", lowercased, so a post and its replies get the same name.
threads = [re.sub(r"^(re:\s*)+", "", subject_of[f].strip(), flags=re.IGNORECASE).lower()
           for f in newsgroups_train.filenames]           ## a list of 11,096 strings

## np.unique sorts the distinct names. return_inverse=True also returns, for every post, where its name sits in that
## sorted list. It returns two arrays; [1] takes the second: one integer thread id per post.
thread_ids = np.unique(threads, return_inverse=True)[1]   ## (11096,)

## StratifiedGroupKFold makes 10 folds that never split a thread across two folds and keep each fold's class mix
## close to the whole set's. .split(...) produces (train indices, dev indices) once per fold; next(...) takes the first.
## The names like n_splits=10 are keyword arguments: named parameters, so their order doesn't matter.
splitter = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=423)
train_idx, dev_idx = next(splitter.split(newsgroups_train.data, y_train, groups=thread_ids))   ## (9987,), (1109,)

## Select posts by index. A Python list needs a list comprehension; a numpy array accepts an array of indices
## directly (y_train[train_idx] = the elements of y_train at those positions).
X_tr, y_tr = [newsgroups_train.data[i] for i in train_idx], y_train[train_idx]   ## 9,987 strings, (9987,)
X_dev, y_dev = [newsgroups_train.data[i] for i in dev_idx], y_train[dev_idx]     ## 1,109 strings, (1109,)
print(f"{len(set(threads)):,} threads. Training part: {len(X_tr):,} posts, dev: {len(X_dev):,} posts, "
      f"threads in both: {len(set(thread_ids[train_idx]) & set(thread_ids[dev_idx]))}")   ## & = set intersection

5,396 threads. Training part: 9,987 posts, dev: 1,109 posts, threads in both: 0


## Tokenizing and batching

RoBERTa reads token ids, not text. Its tokenizer splits each post into subword tokens (byte-pair encoding, from Lecture 3), puts `<s>` at the start and `</s>` at the end, and cuts posts longer than 512 tokens, RoBERTa's limit. That's the same first-512-token window the frozen notebooks used.

- **We tokenize inside the DataLoader, one batch at a time,** and pad only up to the longest post in that batch. The `attention_mask` (1 = real token, 0 = padding) tells RoBERTa to ignore the padding.
- **Training batches group posts of similar length**, in a new random order every epoch. A batch is padded to its longest post, so a random batch that mixes a 30-token post with a 512-token one pads all 16 posts to 512. Random batches pad to 2.6× the tokens that length-grouped ones do, so grouping halves the training time (about 40 s per epoch instead of 80). fairseq, RoBERTa's own code, batches this way, and so does Hugging Face's `Trainer` with `group_by_length=True`.
- Dev and test don't need shuffling or grouping.

**Shapes.** From here on, the comments give the shape of every tensor.
- **B** = posts in a batch (16 in training, 64 in evaluation; the last batch can be smaller)
- **T** = tokens in the batch's longest post, at most 512 (shorter posts are padded up to T)
- **768** = the size of RoBERTa's vectors; **20** = classes

In [3]:
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

## Fix the random number generator, so that every run shuffles the batches the same way, starts the head from the
## same weights and drops out the same values. (Some GPU arithmetic isn't bit-exact, so runs can still differ slightly.)
torch.manual_seed(423)

device = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LENGTH = 512
tokenizer = AutoTokenizer.from_pretrained("roberta-base")  ## loads RoBERTa's vocabulary of 50,265 subword tokens

## collate turns a list of B (text, label) pairs into the 3 tensors that one step needs.
def collate(batch):
    ## batch = [(text0, label0), (text1, label1), ...]. zip(*batch) "unzips" it: the * passes each pair as a separate
    ## argument, and zip groups all the first elements and all the second elements:
    ## texts = (text0, text1, ...), labels = (label0, label1, ...). "a, b = ..." assigns both at once.
    texts, labels = zip(*batch)
    ## The tokenizer returns a dict-like object; with return_tensors="pt" its values are PyTorch tensors.
    enc = tokenizer(list(texts), padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    ## enc["input_ids"]      (B, T) int64: token ids, each row starting with <s> (id 0) and padded with <pad> (id 1)
    ## enc["attention_mask"] (B, T) int64: 1 for a real token, 0 for padding
    return enc["input_ids"], enc["attention_mask"], torch.tensor(labels)   ## + labels (B,) int64

## Each training post's length in tokens (at most 512). Without padding, the tokenizer returns one list of ids per post.
train_lengths = torch.tensor([len(ids) for ids in tokenizer(X_tr, truncation=True, max_length=MAX_LENGTH)["input_ids"]])

## A batch sampler tells the DataLoader which items go in each batch, as lists of indices. This one groups posts of
## similar length. The DataLoader needs only __len__ and __iter__ (like a C++ class with size() and begin()/end()).
class LengthGroupedBatches:
    def __init__(self, lengths, batch_size):
        self.lengths, self.batch_size = lengths, batch_size

    def __len__(self):   ## batches per epoch; len(train_loader) asks this
        return -(-len(self.lengths) // self.batch_size)   ## ceiling division: -(-a // b)

    ## __iter__ runs once per epoch. "yield" hands back one batch and pauses here until the DataLoader asks for the
    ## next one (a generator, like a C++20 coroutine).
    def __iter__(self):
        order = torch.randperm(len(self.lengths))   ## every post, in a new random order: (9987,)
        batches = []
        ## Cut that order into "megabatches" of 50 batches (800 posts), sort each one by length (longest first) and
        ## cut it into batches. Sorting within a random 800, not the whole set, keeps the batches different every epoch.
        ## .split(n) cuts a tensor into pieces of n (the last piece can be shorter).
        for mega in order.split(50 * self.batch_size):
            mega = mega[self.lengths[mega].argsort(descending=True)]
            batches += mega.split(self.batch_size)   ## += appends the pieces: a list of (16,) index tensors
        for i in torch.randperm(len(batches)):      ## hand the batches out in a random order
            yield batches[i].tolist()

## A DataLoader serves batches: it asks the batch sampler which items form each batch and calls collate on each
## group. (Without a batch sampler, shuffle=True and batch_size=16 would make random batches.) A plain list of
## (text, label) pairs is enough as the dataset.
train_loader = DataLoader(list(zip(X_tr, y_tr)), batch_sampler=LengthGroupedBatches(train_lengths, 16),
                          collate_fn=collate)                                                          ## 625 batches
dev_loader = DataLoader(list(zip(X_dev, y_dev)), batch_size=64, collate_fn=collate)                    ## 18 batches
test_loader = DataLoader(list(zip(newsgroups_test.data, y_test)), batch_size=64, collate_fn=collate)   ## 116 batches

## Look at one training batch: iter(...) starts a pass over the loader, next(...) takes its first batch
input_ids, attention_mask, labels = next(iter(train_loader))
print("token ids:", tuple(input_ids.shape), " attention mask:", tuple(attention_mask.shape), " labels:", tuple(labels.shape))
## input_ids[0, :10] = row 0, columns 0 to 9: the first 10 token ids of the first post, (10,). .tolist() makes it a Python list.
## In RoBERTa's tokens, Ġ marks a space before the token and Ċ is a newline.
print("first tokens of the first post:", tokenizer.convert_ids_to_tokens(input_ids[0, :10].tolist()))

/home/tonyxdsu/venvs/cs423/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


token ids: (16, 186)  attention mask: (16, 186)  labels: (16,)
first tokens of the first post: ['<s>', 'M', 'IGHT', 'Y', 'ĠON', 'ES', 'ĠGET', 'ĠM', 'IGHT', 'IER']


## The model: RoBERTa + Hugging Face's classification head on `<s>`

- **The encoder** is the same pre-trained `roberta-base` the frozen notebooks used (12 layers, 768-d vectors).
- **The head** copies Hugging Face's `RobertaClassificationHead`: dropout → Linear(768, 768) → tanh → dropout → Linear(768, 20). Both linear layers are new and randomly initialized. Both dropouts are 0.1, Hugging Face's default for `roberta-base`.
- **Why this head?** An earlier version of this notebook used a single linear layer (dropout → Linear(768, 20)), as in the BERT paper. It scored 72.84 on test (`fine-tuning-results.md`). The extra 768 → 768 tanh layer has the same shape as BERT's pooler, but here it starts from random weights. RoBERTa's own pooler is left out (`add_pooling_layer=False`), since RoBERTa never trained it on anything. RoBERTa's 12 layers adapt either way, so expect a small difference, possibly within seed noise.

Hugging Face's `AutoModelForSequenceClassification` builds the same model in one line. Writing it out shows there's nothing more to it.


In [4]:
from transformers import AutoModel

## A model is a class derived from nn.Module, like `class RobertaClassifier : public nn::Module` in C++.
## Every layer stored as an attribute in __init__ (self.something = layer) is registered automatically: its weights
## show up in model.parameters(), so the optimizer can train them.
class RobertaClassifier(nn.Module):
    def __init__(self, n_classes):    ## the constructor. "self" is C++'s "this", but written out as the first parameter
        super().__init__()            ## call the base class's constructor first (it sets up that registration)
        ## The pre-trained encoder: the same weights the frozen notebooks used. add_pooling_layer=False leaves out
        ## RoBERTa's extra "pooler" layer on top of <s>, which we don't use.
        self.roberta = AutoModel.from_pretrained("roberta-base", add_pooling_layer=False)
        self.dropout = nn.Dropout(0.1)          ## huggingface's default dropout for roberta-base is 0.1, so we use that here too
        self.head = nn.Sequential(              ## runs its layers in order, so self.head(x) acts like one layer
            nn.Linear(768, 768), nn.Tanh(),     ## new and random: (B, 768) -> (B, 768); tanh squashes each value into (-1, 1)
            nn.Dropout(0.1),                    ## the same rate again, as Hugging Face does
            nn.Linear(768, n_classes))          ## new and random: weight (20, 768), bias (20,); (B, 768) -> (B, 20)

    ## forward defines what model(input_ids, attention_mask) computes: calling the object runs forward, like
    ## operator() in C++. There's no backward to write: PyTorch records each operation in forward and derives the
    ## backward pass itself (autograd).
    def forward(self, input_ids, attention_mask):    ## both (B, T)
        ## Inside self.roberta, to draw it out:
        ##   input_ids (B, T) -> each id's row of a (50265, 768) table, plus a position vector, LayerNorm -> (B, T, 768)
        ##   then 12 layers, each (B, T, 768) -> (B, T, 768):
        ##     self-attention: Q, K, V = three 768x768 projections, each (B, T, 768), split into 12 heads: (B, 12, T, 64)
        ##                     scores = Q Kᵀ / 8 : (B, 12, T, T), padding masked out, softmax over the last axis
        ##                     scores @ V : (B, 12, T, 64) -> heads joined: (B, T, 768) -> a 768x768 projection
        ##                     + the layer's input (residual connection), LayerNorm
        ##     feed-forward:   (B, T, 768) -> Linear -> (B, T, 3072) -> GELU -> Linear -> (B, T, 768), + input, LayerNorm
        ##   -> last_hidden_state (B, T, 768): one vector per token, each mixed with the whole post through attention
        hidden = self.roberta(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state   ## (B, T, 768)
        ## [:, 0] = all B posts (":" = everything along axis 0), token 0 along axis 1 (<s>), and all of axis 2.
        ## Indexing one position drops that axis: (B, T, 768) -> (B, 768).
        s_vector = hidden[:, 0]                       ## (B, 768): the <s> vector, as in the frozen notebooks
        return self.head(self.dropout(s_vector))      ## (B, 768) -> dropout -> head -> (B, 20): one score per class

model = RobertaClassifier(n_classes).to(device)   ## build it, then .to(device) moves all its weights to GPU memory
## model.parameters() yields every weight tensor; p.numel() is its number of elements (the product of its shape).
## sum(... for p in ...) adds them up without building a list first.
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters, all of them trainable")

124.7M parameters, all of them trainable


## Optimizer and learning-rate schedule

The RoBERTa paper's fine-tuning settings (Liu et al. 2019, Table 10), with the Adam settings RoBERTa was pre-trained with:
- **AdamW, learning rate 2e-5, weight decay 0.1.** RoBERTa's weights only need nudging, so the rate is far smaller than the frozen MLP's 1e-4. Weight decay pulls every weight slightly toward 0 at each step, a regularizer.
- **Adam β₂ = 0.98 and ε = 1e-6** (PyTorch's defaults are 0.999 and 1e-8). β₂ sets how long Adam's running average of squared gradients remembers: 0.98 forgets faster, so the step sizes adapt faster.
- **Warmup, then linear decay.** The learning rate starts at 0, rises to 2e-5 over the first 6% of steps, then falls back to 0 by the end.
  - At the start the head is random, so the gradients flowing back into RoBERTa are large and not yet meaningful. Warmup keeps those first steps small.
  - Decaying to 0 lets the weights settle at the end.
- **10 epochs**, keeping the epoch with the best dev accuracy (see the training loop).


In [5]:
from transformers import get_linear_schedule_with_warmup

EPOCHS = 10
LR = 2e-5
## AdamW keeps two running averages for every weight (of its gradient, and of its squared gradient), so it holds two
## extra tensors per weight tensor, with the same shapes: about 2 x 124M more numbers in GPU memory.
## model.parameters() includes all of RoBERTa's weights, so they all get trained: that's what makes this fine-tuning.
## betas=(0.9, 0.98): how quickly the two running averages forget (β₁ for the gradient, β₂ for its square).
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.1, betas=(0.9, 0.98), eps=1e-6)
total_steps = EPOCHS * len(train_loader)   ## 10 x 625 = 6250 (one step = one batch)
warmup_steps = int(0.06 * total_steps)     ## 375; int() rounds down
## The scheduler sets the optimizer's learning rate each time scheduler.step() is called: 0 -> 2e-5 over the first
## 375 steps, then back down to 0 by the last step.
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
## Cross-entropy takes logits (B, 20) and labels (B,): it softmaxes each row into 20 probabilities and averages
## -log(probability of the true class) over the B posts. The result is a single number.
criterion = nn.CrossEntropyLoss()
print(f"{len(train_loader)} batches per epoch, {total_steps} steps in total, {warmup_steps} of them warmup")

625 batches per epoch, 6250 steps in total, 375 of them warmup


## One step in slow motion

Before the real loop, here's one batch going through the forward pass, the loss and the backward pass, printing every shape. It doesn't change any weights: there's no `optimizer.step()`, and the gradients are thrown away at the end.

In [6]:
## The example batch from above. .to(device) returns a copy in GPU memory (like a cudaMemcpy to the device).
ids, mask, y_b = input_ids.to(device), attention_mask.to(device), labels.to(device)
print("input_ids      ", tuple(ids.shape), ids.dtype)      ## (B, T) int64
print("attention_mask ", tuple(mask.shape), mask.dtype)    ## (B, T) int64
print("labels         ", tuple(y_b.shape), y_b.dtype)      ## (B,) int64

model.eval()   ## dropout off, so the demo comes out the same every time
## Forward pass, one piece at a time (model(ids, mask) would do all three lines at once).
## autocast: inside this with-block, the big matrix multiplications run in bfloat16, as in training.
with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
    hidden = model.roberta(input_ids=ids, attention_mask=mask).last_hidden_state
    s_vector = hidden[:, 0]
    logits = model.head(model.dropout(s_vector))
## hidden is float32 even inside autocast: RoBERTa's last step is a LayerNorm, which autocast keeps in 32-bit for
## accuracy. Only matrix multiplications, like the head's, run in bfloat16.
print("hidden         ", tuple(hidden.shape), hidden.dtype)      ## (B, T, 768)
print("s_vector       ", tuple(s_vector.shape), s_vector.dtype)  ## (B, 768)
print("logits         ", tuple(logits.shape), logits.dtype)      ## (B, 20), bfloat16 because of autocast

## Loss. .float() converts the logits to 32-bit first. The result is a 0-dimensional tensor: shape () = one number.
## With a random head the 20 probabilities are all close to 1/20, so the loss starts near -log(1/20) = 3.0.
loss = criterion(logits.float(), y_b)
print("loss           ", tuple(loss.shape), round(loss.item(), 3))   ## .item() copies it into a plain Python float

## Backward pass (backpropagation): for every weight w, store d(loss)/d(w) in w.grad, which has w's exact shape
loss.backward()
## Sequential numbers its layers 0-3 (Linear, Tanh, Dropout, Linear); [3] is the last one, 768 -> 20
w = model.head[3].weight
print("head[3].weight ", tuple(w.shape), "  its .grad", tuple(w.grad.shape))
w = model.roberta.encoder.layer[0].intermediate.dense.weight   ## layer 1's feed-forward 768 -> 3072
print("layer 1 FFN in ", tuple(w.shape), "  its .grad", tuple(w.grad.shape))
w = model.roberta.embeddings.word_embeddings.weight
## w.grad.abs().sum(dim=1): the size of each row's gradient, (50265,). "> 0" makes a True/False per row, and the
## outer .sum() counts the Trues: only the rows of tokens that appear in this batch get a gradient.
n_rows = int((w.grad.abs().sum(dim=1) > 0).sum())
print("word embeddings", tuple(w.shape), "  its .grad", tuple(w.grad.shape), f"  ({n_rows:,} rows nonzero)")

model.zero_grad(set_to_none=True)   ## throw the gradients away: this was only a demo

input_ids       (16, 186) torch.int64
attention_mask  (16, 186) torch.int64
labels          (16,) torch.int64


hidden          (16, 186, 768) torch.float32
s_vector        (16, 768) torch.float32
logits          (16, 20) torch.bfloat16
loss            () 2.964
head[3].weight  (20, 768)   its .grad (20, 768)
layer 1 FFN in  (3072, 768)   its .grad (3072, 768)
word embeddings (50265, 768)   its .grad (50265, 768)   (1,163 rows nonzero)


## Evaluation

Same as in the frozen notebooks: `model.eval()` turns dropout off, and `torch.no_grad()` skips storing activations, since there's no backward pass.

**bfloat16:** inside `torch.autocast`, the big matrix multiplications run in 16-bit numbers. That's about twice as fast and takes half the memory. The weights themselves stay 32-bit, because bfloat16 keeps only 2 to 3 significant digits:
- Near 0.05, the closest bfloat16 numbers are about 0.00024 apart.
- One AdamW step at learning rate 2e-5 moves a weight by roughly 0.00002.
- Stored in bfloat16, that update would round away to nothing.

In [7]:
## Returns (mean loss, accuracy, predicted classes) over all the batches of one DataLoader.
def evaluate(loader):
    model.eval()   ## dropout off; the model stays in evaluation mode until model.train() is called
    total_loss, correct, n, preds = 0.0, 0, 0, []   ## several assignments at once; [] is an empty list
    ## A with-block switches something on for its body and back off at the end, like an RAII guard object in C++.
    ## torch.no_grad(): don't record operations for a backward pass (there's none here), which saves memory and time.
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        for input_ids, attention_mask, labels in loader:   ## (B, T), (B, T), (B,)
            input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)
            logits = model(input_ids, attention_mask)       ## (B, 20); calling the model runs forward
            ## criterion gives the batch's mean loss; times B gives its total, so that a smaller last batch
            ## counts for less. .item() turns the 0-dim tensor into a Python float.
            total_loss += criterion(logits.float(), labels).item() * len(labels)
            pred = logits.argmax(dim=1)                      ## (B,): the column of each row's largest logit = predicted class
            correct += (pred == labels).sum().item()         ## (pred == labels) is (B,) True/False; .sum() counts the Trues
            n += len(labels)
            preds.append(pred.cpu())                         ## keep this batch's predictions, copied back to CPU memory
    ## torch.cat joins the list of (B,) tensors end to end into one (N,) tensor; .numpy() makes it a numpy array
    return total_loss / n, correct / n, torch.cat(preds).numpy()

## Training loop

Each step is the same as in the MLP notebooks (zero the gradients, forward, loss, backward, step), plus two lines:
- **`clip_grad_norm_(..., 1.0)`:** if one batch produces an unusually large gradient, it's scaled down so that a single step can't move the weights too far. This is standard when fine-tuning transformers.
- **`scheduler.step()`:** moves the learning rate one step along the warmup-then-decay schedule.

After each epoch we score the dev set and keep a copy of the weights from the epoch with the **highest dev accuracy**, as the RoBERTa recipe does. Over 10 epochs, dev loss is a poor guide: after epoch 2 or so it rises while dev accuracy still improves. The model grows overconfident, so each wrong answer costs more loss even as it gets more answers right.

In [8]:
import time

## best remembers the best epoch so far: a dict with string keys. None is Python's null.
best = {"dev_acc": 0.0, "epoch": 0, "state": None}
start = time.perf_counter()   ## a clock, in seconds

for epoch in range(1, EPOCHS + 1):   ## epoch = 1, 2, ..., 10 (range stops before its end, like i < EPOCHS + 1)
    model.train()                    ## dropout on
    train_loss = 0.0
    ## enumerate(..., start=1) yields (1, batch), (2, batch), ...; each batch is unpacked into its three tensors
    for step, (input_ids, attention_mask, labels) in enumerate(train_loader, start=1):
        ## input_ids, attention_mask: (B, T), with B = 16 (3 in one batch per epoch) and T <= 512 different in every batch
        ## labels: (B,)
        input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)

        ## 1. Clear the gradients: backward() ADDS to each weight's .grad (+=), so the last step's must go first.
        optimizer.zero_grad()

        ## 2. Forward pass, in bfloat16. PyTorch records every operation (the "graph") so it can go backward later.
        ##    (B, T) ids -> RoBERTa -> (B, T, 768) -> <s> -> (B, 768) -> head -> (B, 20)
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            logits = model(input_ids, attention_mask)   ## (B, 20)

        ## 3. Loss: one number for the whole batch, the mean over its B posts
        loss = criterion(logits.float(), labels)        ## ()

        ## 4. Backward pass: from the loss back through the head and all 12 layers. Every weight gets its
        ##    d(loss)/d(weight) in .grad, with the weight's shape: head[3].weight.grad (20, 768), each layer's
        ##    feed-forward weights (3072, 768) and (768, 3072), and so on, about 124M numbers in all.
        loss.backward()

        ## 5. Clip: treat all those ~124M gradient numbers as one long vector. If its length (L2 norm) is over 1.0,
        ##    scale every gradient down so the length is exactly 1.0. A trailing _ in a PyTorch name means "in place".
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        ## 6. Update every weight from its .grad. AdamW, for each weight:
        ##    weight -= learning rate * (running average of its gradient) / (running typical size of its gradient)
        ##    weight -= learning rate * 0.01 * weight   (weight decay)
        optimizer.step()

        ## 7. Move the learning rate one step along the warmup-then-decay schedule
        scheduler.step()

        train_loss += loss.item() * len(labels)   ## .item() also makes the CPU wait until the GPU has finished this step
        if step % 100 == 0:                       ## % is modulo, as in C++: print every 100 steps
            print(f"  epoch {epoch}, step {step}/{len(train_loader)}: loss {loss.item():.3f}, "
                  f"learning rate {scheduler.get_last_lr()[0]:.1e}")

    ## End of the epoch: score the dev set. evaluate() switches to eval mode; the next epoch's model.train() switches back.
    dev_loss, dev_acc, _ = evaluate(dev_loader)   ## _ = a returned value we don't need (the predictions)
    print(f"epoch {epoch}: train loss {train_loss / len(X_tr):.3f}  dev loss {dev_loss:.3f}  dev acc {dev_acc*100:.2f}  "
          f"({time.perf_counter() - start:.0f} s so far)")
    if dev_acc >= best["dev_acc"]:   ## >=: on a tie, keep the later epoch
        ## model.state_dict() is a dict from each weight's name (like "head.3.weight") to its tensor. The dict
        ## comprehension copies every tensor: .detach() = without its gradient history, .cpu() = into CPU memory
        ## (the GPU needs its memory for training), .clone() = a real copy that later training can't change.
        best = {"dev_acc": dev_acc, "epoch": epoch,
                "state": {name: w.detach().cpu().clone() for name, w in model.state_dict().items()}}

## Copy the best epoch's weights back into the model (back onto the GPU)
model.load_state_dict(best["state"])
print(f"Using epoch {best['epoch']} (highest dev accuracy). Peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

  epoch 1, step 100/625: loss 3.045, learning rate 5.3e-06


  epoch 1, step 200/625: loss 2.301, learning rate 1.1e-05


  epoch 1, step 300/625: loss 1.395, learning rate 1.6e-05


  epoch 1, step 400/625: loss 1.168, learning rate 2.0e-05


  epoch 1, step 500/625: loss 0.900, learning rate 2.0e-05


  epoch 1, step 600/625: loss 1.593, learning rate 1.9e-05


... (60 outputs left out)

  epoch 10, step 400/625: loss 0.002, learning rate 7.7e-07


  epoch 10, step 500/625: loss 0.002, learning rate 4.3e-07


  epoch 10, step 600/625: loss 1.186, learning rate 8.5e-08


epoch 10: train loss 0.065  dev loss 1.432  dev acc 76.92  (464 s so far)
Using epoch 3 (highest dev accuracy). Peak GPU memory: 5.2 GB


## Test set, once

In [9]:
from sklearn import metrics

test_loss, test_acc, test_pred = evaluate(test_loader)   ## test_pred: (7370,), the predicted class of every test post
## Macro-F1: each class's F1 score, then their plain average, so small classes count as much as big ones
print(f"Fine-tuned RoBERTa test accuracy: {test_acc*100:.2f}  "
      f"(macro-F1 {metrics.f1_score(y_test, test_pred, average='macro')*100:.2f})\n")

## A list of (name, accuracy) tuples, printed as a table. {name:57s} pads the name to 57 characters (like %-57s).
rows = [("RoBERTa <s>, frozen + LR (tuned)", 67.23),
        ("RoBERTa <s>, frozen + MLP (tuned, mean of 10 seeds)", 68.33),
        ("RoBERTa, fine-tuned, linear head, 3 epochs (earlier run)", 72.84),
        ("RoBERTa, fine-tuned, HF head, 3 epochs (earlier run)", 71.26),
        ("RoBERTa, fine-tuned, HF head, 10 epochs (this notebook)", test_acc * 100),
        ("Qwen3-Embedding-4B, frozen + MLP (10-seed ensemble)", 78.97)]
for name, acc in rows:
    print(f"{name:57s} {acc:6.2f}")
print(f"(1 point = {len(y_test) / 100:.0f} test messages)")

Fine-tuned RoBERTa test accuracy: 71.37  (macro-F1 69.22)

RoBERTa <s>, frozen + LR (tuned)                           67.23
RoBERTa <s>, frozen + MLP (tuned, mean of 10 seeds)        68.33
RoBERTa, fine-tuned, linear head, 3 epochs (earlier run)   72.84
RoBERTa, fine-tuned, HF head, 3 epochs (earlier run)       71.26
RoBERTa, fine-tuned, HF head, 10 epochs (this notebook)    71.37
Qwen3-Embedding-4B, frozen + MLP (10-seed ensemble)        78.97
(1 point = 74 test messages)


**Result:** with the RoBERTa paper's recipe, this run scored 71.37 on test. It kept epoch 3, where dev accuracy happened to peak (77.55), before the learning rate had decayed.
- **The recipe helps a little, on average.** `sweep.py` ran the same recipe with 3 other seeds: 72.84, 73.11 and 73.45 (keeping epochs 8, 8 and 9). Over all 4 runs the recipe averages 72.69, against 72.02 for the 4 runs of the BERT paper's 3-epoch recipe (71.26 to 72.84). That's a gain of under a point, about the size of the seed noise.
- **Ensemble:** averaging the probabilities of `sweep.py`'s 3 models scores **73.93**, the best fine-tuned RoBERTa so far. That's 0.2 below the published RoBERTa-large fine-tune (74.1), and still 5 points behind the frozen Qwen3-Embedding-4B (78.97).
- **Picking the best of 10 dev epochs is noisy.** Dev accuracy moves by about half a point from one epoch to the next, so the "best" epoch can be an early one that happened to land high, as it did here.
- **Dev loss bottomed out at epoch 3** (0.850) and rose to 1.43 by epoch 10 while training loss fell to 0.07: the model memorizes the training set and grows overconfident, yet its dev accuracy holds at 76.5 to 77.5.
- **Dev (77.6) is above test (71.4)** even though the dev set is grouped by thread: dev comes from the same period as train, and the test set is later in time.
- **Cost:** about 8 minutes (46 s per epoch with length-grouped batches, down from 80 s) and 5.2 GB of GPU memory at most.

All runs so far: `fine-tuning-results.md`. For comparison, published fine-tunes in a similar setting (`FrozenResults.md`): RoBERTa-large at 74.1 (Edwards & Camacho-Collados, who don't say how they preprocessed) and DistilBERT at 67.2 (Sada et al.).
